In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# Cell 1 - Environment, imports, reproducibility
import os
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_JAX", "0")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

import csv
import json
import math
import random
import shutil
import sys
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    import librosa
except Exception as e:
    print("librosa import failed, installing...", repr(e))
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "librosa", "soundfile"], check=True)
    import librosa

from transformers import AutoFeatureExtractor, AutoModel, get_cosine_schedule_with_warmup

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
print("platform:", "kaggle" if IS_KAGGLE else ("colab" if IS_COLAB else "local"))
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

# P100 sometimes has fragile cuDNN RNN kernels. This model has no LSTM, but disabling
# cuDNN keeps behavior closer to the stable Kaggle notebooks if needed.
torch.backends.cudnn.enabled = False
print("cuDNN enabled:", torch.backends.cudnn.enabled)

In [ ]:
# Cell 2 - Config
@dataclass
class CFG:
    pretrained_model: str = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"
    seed: int = 3407
    sample_rate: int = 16000
    dev_ratio: float = 0.10

    # A100/Colab default.
    epochs: int = 45
    batch_size: int = 8
    grad_accum_steps: int = 1
    learning_rate: float = 8e-6
    weight_decay: float = 0.01
    warmup_ratio: float = 0.08
    max_grad_norm: float = 1.0

    dropout: float = 0.15
    prompt_layers: int = 2
    adapter_layers: int = 4
    adapter_kernel: int = 5
    num_heads: int = 8
    freeze_feature_extractor: bool = True

    # Online waveform augmentation.
    augment_audio: bool = True
    gain_prob: float = 0.35
    noise_prob: float = 0.25
    speed_prob: float = 0.12
    max_audio_seconds: float = 30.0

    # Dataset-level augmentation.
    # Faster A100 setting: keep only one extra copy for mispronounced rows.
    augment_dataset: bool = True
    aug_copies_all: int = 0
    aug_copies_misp: int = 1

    # Early stopping / checkpoint selection.
    early_stopping_patience: int = 8
    early_stopping_min_delta: float = 1e-4
    early_stopping_metric: str = "dev_wer"  # lower is better

    # Colab/Kaggle path handling.
    work_dir: str = ""
    colab_mount_drive: bool = True
    drive_mount_point: str = "/content/drive"
    save_every: int = 5
    infer_batch_size: int = 1


cfg = CFG()

if IS_COLAB and cfg.colab_mount_drive:
    try:
        from google.colab import drive
        if not Path(cfg.drive_mount_point).exists() or not any(Path(cfg.drive_mount_point).iterdir()):
            print("Mounting Google Drive at", cfg.drive_mount_point)
            drive.mount(cfg.drive_mount_point)
        else:
            print("Google Drive already mounted:", cfg.drive_mount_point)
    except Exception as e:
        print("Drive mount skipped/failed:", repr(e))

if not cfg.work_dir:
    if IS_COLAB:
        cfg.work_dir = "/content/prompt_film_mdd"
    elif IS_KAGGLE:
        cfg.work_dir = "/kaggle/working/prompt_film_mdd"
    else:
        cfg.work_dir = "prompt_film_mdd"


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = Path(cfg.work_dir)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))
print("device:", device)
print("work_dir:", WORK_DIR)

In [ ]:
# Cell 3 - Locate Kaggle/Colab inputs, split data, build vocab
BLANK_TOKEN = "<blank>"
BLANK_ID = 0
PAD_ID = 0


def _glob_many(patterns):
    hits = []
    for pat in patterns:
        try:
            hits.extend(Path("/").glob(pat.lstrip("/")))
        except Exception as e:
            print("glob failed:", pat, repr(e))
    return sorted(set(hits))


def first_existing(patterns, required=True, name="file"):
    hits = _glob_many(patterns)
    if required and not hits:
        print("Could not find", name, "with patterns:")
        for pat in patterns:
            print("  ", pat)

        print("\nKnown dataset variables:")
        for var in [
            "nguyenmanhhust_mdddataset_path",
            "mashwoo_private_mdd_test_path",
        ]:
            if var in globals():
                print(f"  {var} =", globals()[var])

        print("\nTop /content entries:")
        for p in sorted(Path("/content").glob("*"))[:50] if Path("/content").exists() else []:
            print("  ", p)

        print("\nTop kagglehub dataset entries:")
        root = Path("/root/.cache/kagglehub/datasets")
        if root.exists():
            for p in sorted(root.glob("*/*"))[:80]:
                print("  ", p)

        raise FileNotFoundError(name)
    return hits[0] if hits else None


# If Colab's Kaggle import cell created these variables, use them directly.
KAGGLEHUB_TRAIN_ROOT = None
KAGGLEHUB_PRIVATE_ROOT = None

if "nguyenmanhhust_mdddataset_path" in globals():
    KAGGLEHUB_TRAIN_ROOT = Path(nguyenmanhhust_mdddataset_path)
    print("Using KaggleHub train dataset:", KAGGLEHUB_TRAIN_ROOT)

if "mashwoo_private_mdd_test_path" in globals():
    KAGGLEHUB_PRIVATE_ROOT = Path(mashwoo_private_mdd_test_path)
    print("Using KaggleHub private dataset:", KAGGLEHUB_PRIVATE_ROOT)


train_patterns = [
    "/kaggle/input/**/metadata/train_phones.csv",
    "/kaggle/input/**/train_phones.csv",
    "/content/**/metadata/train_phones.csv",
    "/content/**/train_phones.csv",
    "/content/drive/MyDrive/**/metadata/train_phones.csv",
    "/content/drive/MyDrive/**/train_phones.csv",
    "/root/.cache/kagglehub/datasets/**/metadata/train_phones.csv",
    "/root/.cache/kagglehub/datasets/**/train_phones.csv",
]

private_patterns = [
    "/kaggle/input/**/metadata/private_test_submission.csv",
    "/kaggle/input/**/private_test_submission.csv",
    "/content/**/metadata/private_test_submission.csv",
    "/content/**/private_test_submission.csv",
    "/content/drive/MyDrive/**/metadata/private_test_submission.csv",
    "/content/drive/MyDrive/**/private_test_submission.csv",
    "/root/.cache/kagglehub/datasets/**/metadata/private_test_submission.csv",
    "/root/.cache/kagglehub/datasets/**/private_test_submission.csv",
]


if KAGGLEHUB_TRAIN_ROOT is not None:
    train_patterns = [
        str(KAGGLEHUB_TRAIN_ROOT / "**/metadata/train_phones.csv"),
        str(KAGGLEHUB_TRAIN_ROOT / "**/train_phones.csv"),
    ] + train_patterns

if KAGGLEHUB_PRIVATE_ROOT is not None:
    private_patterns = [
        str(KAGGLEHUB_PRIVATE_ROOT / "**/metadata/private_test_submission.csv"),
        str(KAGGLEHUB_PRIVATE_ROOT / "**/private_test_submission.csv"),
    ] + private_patterns


# Labeled public-test metadata is intentionally excluded from the public pipeline.
PUBLIC_CSV = None
PUBLIC_ROOT = None

TRAIN_CSV = first_existing(train_patterns, name="train_phones.csv")
TRAIN_ROOT = TRAIN_CSV.parent.parent if TRAIN_CSV.parent.name == "metadata" else TRAIN_CSV.parent

PRIVATE_CSV = first_existing(private_patterns, required=False, name="private_test_submission.csv")
PRIVATE_ROOT = PRIVATE_CSV.parent.parent if PRIVATE_CSV and PRIVATE_CSV.parent.name == "metadata" else (PRIVATE_CSV.parent if PRIVATE_CSV else None)


print("TRAIN_CSV :", TRAIN_CSV)
print("TRAIN_ROOT:", TRAIN_ROOT)
print("PRIVATE_CSV:", PRIVATE_CSV)
print("PRIVATE_ROOT:", PRIVATE_ROOT)
print("PUBLIC_CSV:", PUBLIC_CSV)
print("PUBLIC_ROOT:", PUBLIC_ROOT)


def read_csv_rows(path: Path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def phone_tokens(text):
    return [tok for tok in str(text).replace("*", " ").split() if tok and tok != "$" and tok != BLANK_TOKEN]


def build_vocab(rows):
    seen = set()
    tokens = []
    for row in rows:
        for col in ("canonical", "transcript"):
            if col not in row:
                continue
            for tok in phone_tokens(row[col]):
                if tok not in seen:
                    seen.add(tok)
                    tokens.append(tok)
    vocab = {BLANK_TOKEN: BLANK_ID}
    vocab.update({tok: i + 1 for i, tok in enumerate(tokens)})
    return vocab


rows = read_csv_rows(TRAIN_CSV)
print("train rows:", len(rows))
print("columns:", rows[0].keys())
print("sample row:", rows[0])

rng = random.Random(cfg.seed)
shuffled = rows[:]
rng.shuffle(shuffled)
dev_n = max(1, int(len(shuffled) * cfg.dev_ratio)) if cfg.dev_ratio > 0 else 0
dev_rows = shuffled[:dev_n]
train_rows = shuffled[dev_n:]


def is_mispronounced(row):
    return phone_tokens(row.get("canonical", "")) != phone_tokens(row.get("transcript", ""))


def build_augmented_rows(base_rows):
    if not cfg.augment_dataset:
        return base_rows[:]

    augmented = []
    all_dups = 0
    misp_dups = 0

    for row in base_rows:
        augmented.append(dict(row, _aug_copy=0, _aug_kind="original"))

        for copy_idx in range(cfg.aug_copies_all):
            all_dups += 1
            augmented.append(dict(row, _aug_copy=copy_idx + 1, _aug_kind="all"))

        if is_mispronounced(row):
            for copy_idx in range(cfg.aug_copies_misp):
                misp_dups += 1
                augmented.append(dict(row, _aug_copy=copy_idx + 1, _aug_kind="misp"))

    random.Random(cfg.seed + 17).shuffle(augmented)

    print("dataset augmentation enabled:", cfg.augment_dataset)
    print("  originals:", len(base_rows))
    print("  all-row augmented copies:", all_dups)
    print("  mispronounced-only augmented copies:", misp_dups)
    print("  augmented train rows:", len(augmented))

    return augmented


train_aug_rows = build_augmented_rows(train_rows)
vocab = build_vocab(rows)
id2token = {idx: tok for tok, idx in vocab.items()}

print("split train/dev raw:", len(train_rows), len(dev_rows))
print("split train/dev effective:", len(train_aug_rows), len(dev_rows))
print("train mispronounced raw:", sum(is_mispronounced(r) for r in train_rows))
print("dev mispronounced raw:", sum(is_mispronounced(r) for r in dev_rows))
print("vocab size:", len(vocab))
print("first vocab tokens:", list(vocab.items())[:20])


def resolve_wav(row, root):
    raw = row.get("path") or row.get("Path") or row.get("audio_path") or row.get("AudioPath")
    p = Path(str(raw))
    if p.is_absolute():
        return p
    if p.suffix.lower() != ".wav":
        p = p.with_suffix(".wav")
    return root / p


for i in range(min(3, len(rows))):
    p = resolve_wav(rows[i], TRAIN_ROOT)
    print(f"audio check {i}:", p, "exists=", p.exists())

with open(WORK_DIR / "vocab_mdd2025.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

with open(WORK_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, ensure_ascii=False, indent=2)

print("saved vocab/config")

In [ ]:
# Cell 4 - Official-ish metrics and decode helpers

def encode(text, vocab):
    return [vocab[tok] for tok in phone_tokens(text) if tok in vocab]


def greedy_decode(frame_logits, id2token):
    pred_ids = frame_logits.argmax(dim=-1).detach().cpu().tolist()
    collapsed = []
    prev = None
    for idx in pred_ids:
        if idx != prev:
            collapsed.append(idx)
        prev = idx
    return " ".join(id2token.get(i, "") for i in collapsed if i != BLANK_ID and id2token.get(i, ""))


def simple_wer(reference, hypothesis):
    ref = phone_tokens(reference)
    hyp = phone_tokens(hypothesis)
    n, m = len(ref), len(hyp)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[n][m] / max(1, n)


def nw_align(seq1, seq2):
    GAP, MATCH, MISMATCH = -1, 1, -1
    n, m = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): score[i][0] = GAP * i
    for j in range(n + 1): score[0][j] = GAP * j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[j - 1] == seq2[i - 1]: s = MATCH
            elif seq1[j - 1] == "<eps>" or seq2[i - 1] == "<eps>": s = GAP
            else: s = MISMATCH
            score[i][j] = max(score[i - 1][j - 1] + s, score[i - 1][j] + GAP, score[i][j - 1] + GAP)
    a1, a2 = [], []
    i, j = m, n
    while i > 0 and j > 0:
        if seq1[j - 1] == seq2[i - 1]: s = MATCH
        elif seq1[j - 1] == "<eps>" or seq2[i - 1] == "<eps>": s = GAP
        else: s = MISMATCH
        if score[i][j] == score[i - 1][j - 1] + s:
            a1.append(seq1[j - 1]); a2.append(seq2[i - 1]); i -= 1; j -= 1
        elif score[i][j] == score[i][j - 1] + GAP:
            a1.append(seq1[j - 1]); a2.append("<eps>"); j -= 1
        else:
            a1.append("<eps>"); a2.append(seq2[i - 1]); i -= 1
    while j > 0:
        a1.append(seq1[j - 1]); a2.append("<eps>"); j -= 1
    while i > 0:
        a1.append("<eps>"); a2.append(seq2[i - 1]); i -= 1
    a1.reverse(); a2.reverse()
    return a1, a2


def ops(a1, a2):
    out = []
    for r, h in zip(a1, a2):
        if r != "<eps>" and h == "<eps>": out.append("D")
        elif r == "<eps>" and h != "<eps>": out.append("I")
        elif r != h: out.append("S")
        else: out.append("C")
    return out


def align_pair(s1, s2):
    a = phone_tokens(s1)
    b = phone_tokens(s2)
    x, y = nw_align(a, b)
    return x, y, ops(x, y)


def accumulate_counters(ref, human, our):
    ref_seq, human_seq, op_rh = align_pair(ref, human)
    human_seq2, our_seq2, op_ho = align_pair(human, our)
    ref_seq3, our_seq3, op_ro = align_pair(ref, our)
    cor_cor = cor_nocor = 0
    sub_sub = sub_sub1 = sub_nosub = 0
    ins_ins = ins_ins1 = ins_noins = 0
    del_del = del_del1 = del_nodel = 0

    flag = 0
    for i in range(len(ref_seq)):
        if ref_seq[i] == "<eps>":
            continue
        while flag < len(ref_seq3) and ref_seq3[flag] == "<eps>":
            flag += 1
        if flag < len(ref_seq3) and ref_seq[i] == ref_seq3[flag]:
            if op_rh[i] == "D" and op_ro[flag] == "D": del_del += 1
            elif op_rh[i] == "D" and op_ro[flag] != "D" and op_ro[flag] != "C": del_del1 += 1
            elif op_rh[i] == "D" and op_ro[flag] != "D" and op_ro[flag] == "C": del_nodel += 1
            flag += 1

    flag = 0
    for i in range(len(human_seq)):
        if human_seq[i] == "<eps>":
            continue
        while flag < len(human_seq2) and human_seq2[flag] == "<eps>":
            flag += 1
        if flag < len(human_seq2) and human_seq[i] == human_seq2[flag]:
            if op_rh[i] == "C" and op_ho[flag] == "C": cor_cor += 1
            elif op_rh[i] == "C" and op_ho[flag] != "C": cor_nocor += 1
            if op_rh[i] == "S" and op_ho[flag] == "C": sub_sub += 1
            elif op_rh[i] == "S" and op_ho[flag] != "C" and ref_seq[i] != our_seq2[flag]: sub_sub1 += 1
            elif op_rh[i] == "S" and op_ho[flag] != "C" and ref_seq[i] == our_seq2[flag]: sub_nosub += 1
            if op_rh[i] == "I" and op_ho[flag] == "C": ins_ins += 1
            elif op_rh[i] == "I" and op_ho[flag] != "C" and op_ho[flag] != "D": ins_ins1 += 1
            elif op_rh[i] == "I" and op_ho[flag] != "C" and op_ho[flag] == "D": ins_noins += 1
            flag += 1

    _, _, op_per = align_pair(human, our)
    return dict(
        cor_cor=cor_cor, cor_nocor=cor_nocor,
        sub_sub=sub_sub, sub_sub1=sub_sub1, sub_nosub=sub_nosub,
        ins_ins=ins_ins, ins_ins1=ins_ins1, ins_noins=ins_noins,
        del_del=del_del, del_del1=del_del1, del_nodel=del_nodel,
        sub_ho=op_per.count("S"), del_ho=op_per.count("D"), ins_ho=op_per.count("I"), cor_ho=op_per.count("C"),
    )


def compute_all_metrics(canonical_list, transcript_list, predict_list):
    totals = {k: 0 for k in [
        "cor_cor", "cor_nocor", "sub_sub", "sub_sub1", "sub_nosub", "ins_ins", "ins_ins1", "ins_noins",
        "del_del", "del_del1", "del_nodel", "sub_ho", "del_ho", "ins_ho", "cor_ho"
    ]}
    for ref, human, our in zip(canonical_list, transcript_list, predict_list):
        c = accumulate_counters(ref, human, our)
        for k, v in c.items(): totals[k] += v
    TR = totals["sub_sub"] + totals["sub_sub1"] + totals["del_del"] + totals["del_del1"] + totals["ins_ins"] + totals["ins_ins1"]
    FR = totals["cor_nocor"]
    FA = totals["sub_nosub"] + totals["ins_noins"] + totals["del_nodel"]
    DE = totals["sub_sub1"] + totals["del_del1"] + totals["ins_ins1"]
    precision = TR / (TR + FR) if TR + FR else 0.0
    recall = TR / (TR + FA) if TR + FA else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    ref_len = totals["sub_ho"] + totals["del_ho"] + totals["cor_ho"]
    per = (totals["sub_ho"] + totals["del_ho"] + totals["ins_ho"]) / ref_len if ref_len else 0.0
    der = DE / TR if TR else 0.0
    score = 0.5 * f1 + 0.4 * (1 - der) + 0.1 * (1 - per)
    return {"score": score, "f1": f1, "der": der, "per": per, "precision": precision, "recall": recall, "tr": TR, "fr": FR, "fa": FA, "de": DE}

print("metric helpers ready")

In [ ]:
# Cell 5 - Dataset, collate, batch debug
feature_extractor = AutoFeatureExtractor.from_pretrained(cfg.pretrained_model)
feature_extractor.return_attention_mask = True

try:
    import soundfile as sf
except Exception:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile"], check=True)
    import soundfile as sf


class MDDDataset(Dataset):
    def __init__(self, rows, data_root, vocab, augment=False, has_transcript=True):
        self.rows = rows
        self.data_root = Path(data_root)
        self.vocab = vocab
        self.augment = augment
        self.has_transcript = has_transcript
        self.max_samples = int(cfg.max_audio_seconds * cfg.sample_rate)

    def __len__(self):
        return len(self.rows)

    def _resolve(self, row):
        raw = row.get("path") or row.get("Path") or row.get("audio_path") or row.get("AudioPath")
        p = Path(str(raw))
        if p.is_absolute():
            return p
        if p.suffix.lower() != ".wav":
            p = p.with_suffix(".wav")
        return self.data_root / p

    def _load_audio_fast(self, wav_path):
        wav, sr = sf.read(str(wav_path), dtype="float32", always_2d=False)

        if wav.ndim == 2:
            wav = wav.mean(axis=1)

        if sr != cfg.sample_rate:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=cfg.sample_rate).astype(np.float32)

        wav = wav[:self.max_samples].astype(np.float32, copy=False)
        return wav

    def _augment(self, wav):
        if random.random() < cfg.gain_prob:
            wav = wav * random.uniform(0.85, 1.15)

        if random.random() < cfg.noise_prob:
            rms = float(np.sqrt(np.mean(np.square(wav))) + 1e-6)
            noise = np.random.normal(
                0.0,
                rms * random.uniform(0.001, 0.006),
                wav.shape,
            ).astype(np.float32)
            wav = wav + noise

        if random.random() < cfg.speed_prob and len(wav) > cfg.sample_rate // 2:
            rate = random.choice([0.95, 1.05])
            wav = librosa.effects.time_stretch(wav.astype(np.float32), rate=rate).astype(np.float32)

        return np.clip(wav, -1.0, 1.0).astype(np.float32)

    def __getitem__(self, idx):
        row = self.rows[idx]
        wav_path = self._resolve(row)
        wav = self._load_audio_fast(wav_path)

        if self.augment:
            wav = self._augment(wav)

        canonical = row.get("canonical") or row.get("Canonical") or ""
        transcript = row.get("transcript") or row.get("Transcript") or canonical

        return {
            "id": row.get("id", str(idx)),
            "path": row.get("path") or row.get("Path") or str(wav_path),
            "waveform": wav,
            "canonical": canonical,
            "transcript": transcript,
            "canonical_ids": encode(canonical, self.vocab),
            "label_ids": encode(transcript, self.vocab),
        }


def collate_fn(batch):
    waveforms = [b["waveform"] for b in batch]
    wav_lengths = torch.tensor([len(w) for w in waveforms], dtype=torch.long)

    inputs = feature_extractor(
        waveforms,
        sampling_rate=cfg.sample_rate,
        padding=True,
        return_tensors="pt",
        return_attention_mask=True,
    )

    max_can = max(max(len(b["canonical_ids"]) for b in batch), 1)
    max_lab = max(max(len(b["label_ids"]) for b in batch), 1)

    canonical = torch.full((len(batch), max_can), PAD_ID, dtype=torch.long)
    labels = torch.full((len(batch), max_lab), PAD_ID, dtype=torch.long)
    label_lengths = torch.tensor([len(b["label_ids"]) for b in batch], dtype=torch.long)

    for i, b in enumerate(batch):
        if b["canonical_ids"]:
            canonical[i, :len(b["canonical_ids"])] = torch.tensor(b["canonical_ids"], dtype=torch.long)
        if b["label_ids"]:
            labels[i, :len(b["label_ids"])] = torch.tensor(b["label_ids"], dtype=torch.long)

    return {
        "ids": [b["id"] for b in batch],
        "paths": [b["path"] for b in batch],
        "canonical_str": [b["canonical"] for b in batch],
        "transcript_str": [b["transcript"] for b in batch],
        "input_values": inputs.input_values,
        "attention_mask": inputs.attention_mask if hasattr(inputs, "attention_mask") else None,
        "wav_lengths": wav_lengths,
        "canonical_ids": canonical,
        "labels": labels,
        "label_lengths": label_lengths,
    }


train_ds = MDDDataset(train_aug_rows, TRAIN_ROOT, vocab, augment=cfg.augment_audio, has_transcript=True)
dev_ds = MDDDataset(dev_rows, TRAIN_ROOT, vocab, augment=False, has_transcript=True)

print("effective train dataset rows:", len(train_ds), "raw train rows:", len(train_rows))
print("dev dataset rows:", len(dev_ds))

NUM_WORKERS_TRAIN = 4
NUM_WORKERS_DEV = 2

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS_TRAIN,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS_TRAIN > 0,
    prefetch_factor=2 if NUM_WORKERS_TRAIN > 0 else None,
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS_DEV,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS_DEV > 0,
    prefetch_factor=2 if NUM_WORKERS_DEV > 0 else None,
)

print("train loader batches:", len(train_loader), "workers:", NUM_WORKERS_TRAIN)
print("dev loader batches:", len(dev_loader), "workers:", NUM_WORKERS_DEV)

batch = next(iter(train_loader))
print("batch keys:", batch.keys())
print("input_values:", tuple(batch["input_values"].shape))
print("attention_mask:", None if batch["attention_mask"] is None else tuple(batch["attention_mask"].shape))
print("canonical_ids:", tuple(batch["canonical_ids"].shape))
print("labels:", tuple(batch["labels"].shape), "label_lengths:", batch["label_lengths"].tolist())
print("sample canonical:", batch["canonical_str"][0])
print("sample transcript:", batch["transcript_str"][0])

In [ ]:
# Cell 6 - New architecture: Prompt-FiLM CTC
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, dim, 2, dtype=torch.float32) * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class AcousticAdapterBlock(nn.Module):
    def __init__(self, hidden, dropout, kernel):
        super().__init__()
        self.norm1 = RMSNorm(hidden)
        self.dw = nn.Conv1d(hidden, hidden, kernel_size=kernel, padding=kernel // 2, groups=hidden)
        self.pw = nn.Linear(hidden, hidden * 2)
        self.out = nn.Linear(hidden * 2, hidden)
        self.norm2 = RMSNorm(hidden)
        self.ff = nn.Sequential(
            nn.Linear(hidden, hidden * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 4, hidden),
        )
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm1(x)
        h = self.dw(h.transpose(1, 2)).transpose(1, 2)
        h = self.out(F.gelu(self.pw(h)))
        x = x + self.drop(h)
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

class PromptFiLMCTC(nn.Module):
    """Different from the 0.7 notebook.

    The canonical sequence is compressed into a prompt vector. That prompt generates
    FiLM scale/shift/gate values for acoustic adapter blocks. There is no audio-to-
    canonical cross-attention and no linguistic BiLSTM.
    """
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(cfg.pretrained_model)
        hidden = self.backbone.config.hidden_size
        heads = cfg.num_heads
        while hidden % heads != 0 and heads > 1:
            heads -= 1
        self.pad_id = PAD_ID
        self.use_backbone_attention_mask = getattr(self.backbone.config, "feat_extract_norm", "group") == "layer"

        if cfg.freeze_feature_extractor:
            fe = getattr(self.backbone, "feature_extractor", None)
            if fe is not None and hasattr(fe, "_freeze_parameters"):
                fe._freeze_parameters()
                print("feature extractor frozen")

        self.can_emb = nn.Embedding(vocab_size, hidden, padding_idx=PAD_ID)
        self.can_pos = PositionalEncoding(hidden)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=heads,
            dim_feedforward=hidden * 3,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.prompt_encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.prompt_layers)
        self.prompt_proj = nn.Sequential(
            RMSNorm(hidden * 2),
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
        )

        self.adapters = nn.ModuleList([
            AcousticAdapterBlock(hidden, cfg.dropout, cfg.adapter_kernel)
            for _ in range(cfg.adapter_layers)
        ])
        self.film = nn.ModuleList([nn.Linear(hidden, hidden * 2) for _ in range(cfg.adapter_layers)])
        self.gates = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(cfg.adapter_layers)])
        self.final_norm = RMSNorm(hidden)
        self.dropout = nn.Dropout(cfg.dropout)
        self.head = nn.Linear(hidden, vocab_size)

    def get_output_lengths(self, input_lengths):
        if hasattr(self.backbone, "_get_feat_extract_output_lengths"):
            return self.backbone._get_feat_extract_output_lengths(input_lengths).long()
        lengths = input_lengths.to(torch.float)
        for kernel, stride in zip(self.backbone.config.conv_kernel, self.backbone.config.conv_stride):
            lengths = torch.floor((lengths - kernel) / stride + 1)
        return lengths.long()

    def encode_prompt(self, canonical_ids):
        mask = canonical_ids.eq(self.pad_id)
        all_pad = mask.all(dim=1)
        if all_pad.any():
            mask = mask.clone()
            mask[all_pad, 0] = False
        x = self.can_pos(self.can_emb(canonical_ids))
        x = self.prompt_encoder(x, src_key_padding_mask=mask)
        valid = (~mask).unsqueeze(-1).float()
        mean = (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
        x_masked = x.masked_fill(mask.unsqueeze(-1), -1e4)
        maxv = x_masked.max(dim=1).values
        maxv = torch.where(torch.isfinite(maxv), maxv, torch.zeros_like(maxv))
        return self.prompt_proj(torch.cat([mean, maxv], dim=-1))

    def forward(self, input_values, canonical_ids, wav_lengths, attention_mask=None):
        backbone_mask = attention_mask if self.use_backbone_attention_mask else None
        acoustic = self.backbone(input_values, attention_mask=backbone_mask).last_hidden_state
        out_lengths = self.get_output_lengths(wav_lengths).clamp(max=acoustic.size(1))
        prompt = self.encode_prompt(canonical_ids)

        x = acoustic
        for block, film, gate_layer in zip(self.adapters, self.film, self.gates):
            h = block(x)
            scale, shift = film(prompt).unsqueeze(1).chunk(2, dim=-1)
            gate = torch.sigmoid(gate_layer(prompt)).unsqueeze(1)
            h = h * (1.0 + 0.5 * torch.tanh(scale)) + 0.5 * shift
            x = x + gate * h
        logits = self.head(self.dropout(self.final_norm(x)))
        return logits, out_lengths

model = PromptFiLMCTC(cfg, len(vocab)).to(device)
params_total = sum(p.numel() for p in model.parameters())
params_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"model params trainable/total: {params_train:,}/{params_total:,}")

with torch.no_grad():
    b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
    logits, out_lens = model(b["input_values"], b["canonical_ids"], b["wav_lengths"], b["attention_mask"])
print("forward debug logits:", tuple(logits.shape), "out_lens:", out_lens.detach().cpu().tolist())
print("one decoded before train:", greedy_decode(logits[0, :int(out_lens[0])], id2token))

In [ ]:
# Cell 7 - Train / evaluate functions
ctc_loss = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
updates_per_epoch = max(1, math.ceil(len(train_loader) / cfg.grad_accum_steps))
total_steps = max(1, updates_per_epoch * cfg.epochs)
warmup_steps = int(total_steps * cfg.warmup_ratio)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print("updates_per_epoch:", updates_per_epoch, "total_steps:", total_steps, "warmup_steps:", warmup_steps)


def move_batch(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def train_one_epoch(epoch):
    model.train()
    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(train_loader, desc=f"train epoch {epoch}")
    for step, batch in enumerate(pbar, 1):
        batch = move_batch(batch, device)
        logits, out_lens = model(batch["input_values"], batch["canonical_ids"], batch["wav_lengths"], batch["attention_mask"])
        log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
        out_lens = out_lens.clamp(max=log_probs.size(0))
        loss = ctc_loss(log_probs, batch["labels"], out_lens, batch["label_lengths"])
        (loss / cfg.grad_accum_steps).backward()
        losses.append(float(loss.item()))
        if step % cfg.grad_accum_steps == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        if step % 50 == 0 or step == 1:
            pbar.set_postfix(loss=np.mean(losses[-30:]), lr=optimizer.param_groups[0]["lr"])
    return float(np.mean(losses))

@torch.no_grad()
def evaluate_dev(epoch=0, max_print=4):
    model.eval()
    rows_pred = []
    losses = []
    wers = []
    for batch in tqdm(dev_loader, desc="dev"):
        batch = move_batch(batch, device)
        logits, out_lens = model(batch["input_values"], batch["canonical_ids"], batch["wav_lengths"], batch["attention_mask"])
        log_probs = F.log_softmax(logits, dim=-1)
        loss = ctc_loss(log_probs.transpose(0, 1), batch["labels"], out_lens.clamp(max=log_probs.size(1)), batch["label_lengths"])
        losses.append(float(loss.item()))
        for i in range(logits.size(0)):
            valid = min(int(out_lens[i].item()), logits.size(1))
            pred = greedy_decode(logits[i, :valid], id2token)
            can = batch["canonical_str"][i]
            tr = batch["transcript_str"][i]
            rows_pred.append({"id": batch["ids"][i], "canonical": can, "transcript": tr, "predict": pred})
            wers.append(simple_wer(tr, pred))
    metrics = compute_all_metrics(
        [r["canonical"] for r in rows_pred],
        [r["transcript"] for r in rows_pred],
        [r["predict"] for r in rows_pred],
    )
    metrics["dev_ctc"] = float(np.mean(losses)) if losses else 0.0
    metrics["dev_wer"] = float(np.mean(wers)) if wers else 999.0

    print("DEV METRICS", json.dumps({k: round(v, 6) for k, v in metrics.items() if isinstance(v, (int, float))}, ensure_ascii=False))
    print("Sample dev predictions:")
    for r in rows_pred[:max_print]:
        print(" id:", r["id"])
        print(" can:", r["canonical"])
        print(" tru:", r["transcript"])
        print(" pre:", r["predict"])
        print(" wer:", round(simple_wer(r["transcript"], r["predict"]), 4))
    return metrics, rows_pred


def save_ckpt(name, epoch, metrics):
    payload = {
        "arch": "prompt_film_ctc",
        "cfg": asdict(cfg),
        "vocab": vocab,
        "epoch": epoch,
        "metrics": {k: v for k, v in metrics.items() if isinstance(v, (int, float))},
        "model_state_dict": model.state_dict(),
    }
    path = WORK_DIR / name
    torch.save(payload, path)
    print("saved", path)
    return path

In [ ]:
# Cell 8 - Training loop
history = []
best_wer = float("inf")
best_score = -float("inf")
best_wer_path = None
best_score_path = None
early_stop_best = float("inf")
early_stop_bad_epochs = 0

print("early stopping:", {
    "metric": cfg.early_stopping_metric,
    "patience": cfg.early_stopping_patience,
    "min_delta": cfg.early_stopping_min_delta,
})

start = time.time()
for epoch in range(1, cfg.epochs + 1):
    train_loss = train_one_epoch(epoch)
    metrics, dev_predictions = evaluate_dev(epoch)
    record = {"epoch": epoch, "train_loss": train_loss, **{k: v for k, v in metrics.items() if isinstance(v, (int, float))}}
    history.append(record)
    with open(WORK_DIR / "history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    print(
        f"epoch={epoch:03d} train_loss={train_loss:.6f} "
        f"dev_wer={metrics['dev_wer']:.6f} score={metrics['score']:.6f} "
        f"f1={metrics['f1']:.6f} der={metrics['der']:.6f} per={metrics['per']:.6f}"
    )

    if metrics["dev_wer"] < best_wer:
        best_wer = metrics["dev_wer"]
        best_wer_path = save_ckpt("best_wer.pt", epoch, metrics)
        with open(WORK_DIR / "dev_predictions_best_wer.csv", "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["id", "canonical", "transcript", "predict"])
            writer.writeheader(); writer.writerows(dev_predictions)

    if metrics["score"] > best_score:
        best_score = metrics["score"]
        best_score_path = save_ckpt("best_score.pt", epoch, metrics)

    if cfg.save_every and (epoch % cfg.save_every == 0 or epoch == cfg.epochs):
        save_ckpt(f"epoch_{epoch:03d}.pt", epoch, metrics)

    # Early stopping chooses the best model by dev WER. The best checkpoint
    # is still saved before stopping, so later cells always infer from best_wer.pt.
    current_stop_value = float(metrics.get(cfg.early_stopping_metric, metrics["dev_wer"]))
    if current_stop_value < early_stop_best - cfg.early_stopping_min_delta:
        early_stop_best = current_stop_value
        early_stop_bad_epochs = 0
        print("early stop improved:", cfg.early_stopping_metric, early_stop_best)
    else:
        early_stop_bad_epochs += 1
        print("early stop patience:", early_stop_bad_epochs, "/", cfg.early_stopping_patience)
        if cfg.early_stopping_patience > 0 and early_stop_bad_epochs >= cfg.early_stopping_patience:
            print("EARLY STOP at epoch", epoch, "best", cfg.early_stopping_metric, "=", early_stop_best)
            break

print("training done in minutes:", round((time.time() - start) / 60, 2))
print("best_wer:", best_wer, "path:", best_wer_path)
print("best_score:", best_score, "path:", best_score_path)

In [ ]:
# Cell 9 - Inference helpers for public/private
@torch.no_grad()
def infer_rows(test_rows, test_root, ckpt_path=None, out_csv=None):
    if ckpt_path is not None:
        print("loading checkpoint for infer:", ckpt_path)
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    ds = MDDDataset(test_rows, test_root, vocab, augment=False, has_transcript=("transcript" in test_rows[0]))
    loader = DataLoader(ds, batch_size=cfg.infer_batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0)
    outputs = []
    length_diffs = []
    for batch in tqdm(loader, desc="infer"):
        batch = move_batch(batch, device)
        logits, out_lens = model(batch["input_values"], batch["canonical_ids"], batch["wav_lengths"], batch["attention_mask"])
        for i in range(logits.size(0)):
            valid = min(int(out_lens[i].item()), logits.size(1))
            pred = greedy_decode(logits[i, :valid], id2token)
            outputs.append({"id": batch["ids"][i], "path": batch["paths"][i], "predict": pred})
            length_diffs.append(len(phone_tokens(pred)) - len(phone_tokens(batch["canonical_str"][i])))
    print("infer rows:", len(outputs))
    print("length diff mean/min/max:", float(np.mean(length_diffs)), min(length_diffs), max(length_diffs))
    print("sample outputs:")
    for r in outputs[:5]:
        print(r)
    if out_csv:
        with open(out_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["id", "path", "predict"])
            writer.writeheader(); writer.writerows(outputs)
        print("saved csv:", out_csv)
    return outputs

print("Labeled public-test evaluation is disabled; use the held-out dev split.")


In [ ]:
# Cell 10 - Private inference and submission zip
assert PRIVATE_CSV is not None and PRIVATE_ROOT is not None, "Attach private test dataset before running this cell"
private_rows = read_csv_rows(PRIVATE_CSV)
print("private rows:", len(private_rows), "columns:", private_rows[0].keys())

# Default: use best dev WER, following the lesson from the 0.7 notebook.
results_csv = WORK_DIR / "results.csv"
private_outputs = infer_rows(private_rows, PRIVATE_ROOT, ckpt_path=WORK_DIR / "best_wer.pt", out_csv=results_csv)

# Format checks.
assert len(private_outputs) == len(private_rows), (len(private_outputs), len(private_rows))
ids_in = [r["id"] for r in private_rows]
ids_out = [r["id"] for r in private_outputs]
assert ids_in == ids_out, "Output id order differs from private metadata"
assert all(r["predict"] is not None for r in private_outputs)
print("format OK. rows:", len(private_outputs))

zip_path = WORK_DIR / "prediction.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(results_csv, arcname="results.csv")
print("created:", zip_path)
print("zip size bytes:", zip_path.stat().st_size)
print("Kaggle submit file:", zip_path)

# Export a full model bundle too, so the output includes the trained model,
# config, vocab, training history, dev predictions, and the submission zip.
bundle_path = WORK_DIR / "prompt_film_model_bundle.zip"
bundle_files = [
    WORK_DIR / "best_wer.pt",
    WORK_DIR / "best_score.pt",
    WORK_DIR / "vocab_mdd2025.json",
    WORK_DIR / "config.json",
    WORK_DIR / "history.json",
    WORK_DIR / "dev_predictions_best_wer.csv",
    WORK_DIR / "results.csv",
    WORK_DIR / "prediction.zip",
]
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in bundle_files:
        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)
        else:
            print("bundle warning missing:", file_path)
print("created model bundle:", bundle_path)
print("bundle size bytes:", bundle_path.stat().st_size)


# Optional: on Colab, copy the two important zip files to Drive for download/linking.
def copy_outputs_to_drive():
    if not IS_COLAB:
        print("Not Colab; skip Drive copy")
        return
    drive_out = Path(cfg.drive_mount_point) / "MyDrive" / "mdd_prompt_film_outputs"
    drive_out.mkdir(parents=True, exist_ok=True)
    for file_path in [zip_path, bundle_path]:
        if file_path.exists():
            dst = drive_out / file_path.name
            shutil.copy2(file_path, dst)
            print("copied to Drive:", dst)
    print("Drive output folder:", drive_out)

copy_outputs_to_drive()

In [ ]:
# Cell 11 - Confidence / conservative decoders from our own model
from collections import Counter

DECODER_DIR = WORK_DIR / "decoder_candidates"
DECODER_DIR.mkdir(parents=True, exist_ok=True)

CKPT_FOR_DECODER = WORK_DIR / "best_score.pt"
print("decoder checkpoint:", CKPT_FOR_DECODER)


def make_submission_zip(csv_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname="results.csv")
    print("created zip:", zip_path, "bytes:", zip_path.stat().st_size)


def save_prediction_rows(rows, csv_path, zip_path):
    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "path", "predict"])
        writer.writeheader()
        for r in rows:
            writer.writerow({"id": r["id"], "path": r["path"], "predict": r["predict"]})
    make_submission_zip(csv_path, zip_path)


def align_tokens(can, pred):
    n, m = len(can), len(pred)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    bt = [[None] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = i
        bt[i][0] = "D"
    for j in range(1, m + 1):
        dp[0][j] = j
        bt[0][j] = "I"

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub_cost = 0 if can[i - 1] == pred[j - 1] else 1
            opts = [
                (dp[i - 1][j] + 1, "D"),
                (dp[i][j - 1] + 1, "I"),
                (dp[i - 1][j - 1] + sub_cost, "C" if sub_cost == 0 else "S"),
            ]
            dp[i][j], bt[i][j] = min(opts, key=lambda x: x[0])

    out = []
    i, j = n, m
    while i > 0 or j > 0:
        op = bt[i][j]
        if op in ("C", "S"):
            out.append((i - 1, j - 1, can[i - 1], pred[j - 1], op))
            i -= 1
            j -= 1
        elif op == "D":
            out.append((i - 1, None, can[i - 1], None, "D"))
            i -= 1
        else:
            out.append((None, j - 1, None, pred[j - 1], "I"))
            j -= 1

    out.reverse()
    return out


def decode_logits_with_conf(frame_logits, valid_len):
    logits = frame_logits[:valid_len]
    probs = torch.softmax(logits, dim=-1)
    max_prob, pred_ids = probs.max(dim=-1)

    ids = pred_ids.detach().cpu().tolist()
    conf = max_prob.detach().cpu().tolist()

    tokens = []
    token_confs = []
    prev = None
    seg_probs = []

    for idx, p in zip(ids, conf):
        if idx != prev and prev is not None:
            if prev != BLANK_ID:
                tok = id2token.get(prev, "")
                if tok:
                    tokens.append(tok)
                    token_confs.append(float(np.mean(seg_probs)))
            seg_probs = []
        seg_probs.append(p)
        prev = idx

    if prev is not None and seg_probs:
        if prev != BLANK_ID:
            tok = id2token.get(prev, "")
            if tok:
                tokens.append(tok)
                token_confs.append(float(np.mean(seg_probs)))

    return tokens, token_confs


@torch.no_grad()
def infer_details(rows, root, ckpt_path):
    print("loading:", ckpt_path)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    ds = MDDDataset(rows, root, vocab, augment=False, has_transcript=("transcript" in rows[0]))
    loader = DataLoader(ds, batch_size=cfg.infer_batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0)

    details = []
    for batch in tqdm(loader, desc="infer details"):
        batch = move_batch(batch, device)
        logits, out_lens = model(
            batch["input_values"],
            batch["canonical_ids"],
            batch["wav_lengths"],
            batch["attention_mask"],
        )

        for i in range(logits.size(0)):
            valid = min(int(out_lens[i].item()), logits.size(1))
            tokens, confs = decode_logits_with_conf(logits[i], valid)
            pred = " ".join(tokens)
            can = batch["canonical_str"][i]
            can_tokens = phone_tokens(can)

            details.append({
                "id": batch["ids"][i],
                "path": batch["paths"][i],
                "canonical": can,
                "transcript": batch["transcript_str"][i],
                "raw_predict": pred,
                "tokens": tokens,
                "confs": confs,
                "mean_conf": float(np.mean(confs)) if confs else 0.0,
                "min_conf": float(np.min(confs)) if confs else 0.0,
                "len_diff": len(tokens) - len(can_tokens),
            })

    return details


def apply_conf_decoder(row, params):
    can = phone_tokens(row["canonical"])
    pred = row["tokens"]
    confs = row["confs"]

    if not pred:
        return " ".join(can)

    abs_len_diff = abs(len(pred) - len(can))
    rel_len_diff = abs_len_diff / max(1, len(can))

    if (
        abs_len_diff > params["max_abs_len_diff"]
        or rel_len_diff > params["max_rel_len_diff"]
    ) and row["mean_conf"] < params["len_fallback_conf"]:
        return " ".join(can)

    out = []
    for can_i, pred_i, can_tok, pred_tok, op in align_tokens(can, pred):
        pconf = confs[pred_i] if pred_i is not None and pred_i < len(confs) else 0.0

        if op == "C":
            out.append(can_tok)

        elif op == "S":
            if pconf >= params["sub_thr"]:
                out.append(pred_tok)
            else:
                out.append(can_tok)

        elif op == "I":
            if pconf >= params["ins_thr"] and row["mean_conf"] >= params["utt_thr"]:
                out.append(pred_tok)

        elif op == "D":
            if row["mean_conf"] >= params["del_utt_thr"]:
                # Keep deletion.
                pass
            else:
                out.append(can_tok)

    return " ".join(out)


def rows_from_details(details, name, params=None):
    out = []
    for r in details:
        if params is None:
            pred = r["raw_predict"]
        else:
            pred = apply_conf_decoder(r, params)
        out.append({"id": r["id"], "path": r["path"], "predict": pred})
    return out


def diagnostics(name, rows, source_details):
    by_id = {r["id"]: r for r in source_details}
    changed = 0
    len_diffs = []
    empty = 0

    for r in rows:
        d = by_id[r["id"]]
        can = phone_tokens(d["canonical"])
        pred = phone_tokens(r["predict"])
        if pred != can:
            changed += 1
        len_diffs.append(len(pred) - len(can))
        if not pred:
            empty += 1

    print(
        name,
        "| changed_vs_canonical:", changed, "/", len(rows),
        "| empty:", empty,
        "| len_diff mean/min/max:",
        round(float(np.mean(len_diffs)), 4), min(len_diffs), max(len_diffs)
    )


# 1) Infer dev + tune confidence thresholds on dev.
dev_details = infer_details(dev_rows, TRAIN_ROOT, CKPT_FOR_DECODER)

raw_dev_rows = rows_from_details(dev_details, "raw", None)
raw_metrics = compute_all_metrics(
    [d["canonical"] for d in dev_details],
    [d["transcript"] for d in dev_details],
    [r["predict"] for r in raw_dev_rows],
)
print("RAW DEV:", json.dumps(raw_metrics, ensure_ascii=False, indent=2))

grid = []
for sub_thr in [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]:
    for ins_thr in [0.70, 0.80, 0.90]:
        for del_utt_thr in [0.65, 0.75, 0.85, 0.95]:
            grid.append({
                "sub_thr": sub_thr,
                "ins_thr": ins_thr,
                "del_utt_thr": del_utt_thr,
                "utt_thr": 0.60,
                "max_abs_len_diff": 5,
                "max_rel_len_diff": 0.35,
                "len_fallback_conf": 0.80,
            })

scored = []
for params in grid:
    pred_rows = rows_from_details(dev_details, "candidate", params)
    m = compute_all_metrics(
        [d["canonical"] for d in dev_details],
        [d["transcript"] for d in dev_details],
        [r["predict"] for r in pred_rows],
    )
    # Avoid selecting a decoder that kills recall too aggressively.
    objective = m["score"] if m["recall"] >= 0.45 else m["score"] - 0.15
    scored.append((objective, m, params))

scored.sort(key=lambda x: x[0], reverse=True)
print("\nTOP DEV DECODERS:")
for rank, (obj, m, params) in enumerate(scored[:8], 1):
    print(rank, "obj", round(obj, 6), "metrics", {k: round(m[k], 6) for k in ["score", "f1", "der", "per", "precision", "recall"]}, "params", params)

best_params = scored[0][2]
balanced_params = {
    "sub_thr": 0.72,
    "ins_thr": 0.85,
    "del_utt_thr": 0.88,
    "utt_thr": 0.60,
    "max_abs_len_diff": 5,
    "max_rel_len_diff": 0.35,
    "len_fallback_conf": 0.80,
}
strict_params = {
    "sub_thr": 0.82,
    "ins_thr": 0.92,
    "del_utt_thr": 0.94,
    "utt_thr": 0.70,
    "max_abs_len_diff": 4,
    "max_rel_len_diff": 0.28,
    "len_fallback_conf": 0.85,
}

# 2) Infer private once with details, then write candidates.
private_details = infer_details(private_rows, PRIVATE_ROOT, CKPT_FOR_DECODER)

candidate_specs = [
    ("ind_raw_best_score", None),
    ("ind_dev_tuned_conf", best_params),
    ("ind_balanced_conf", balanced_params),
    ("ind_strict_conf", strict_params),
]

for name, params in candidate_specs:
    pred_rows = rows_from_details(private_details, name, params)
    diagnostics(name, pred_rows, private_details)

    csv_path = DECODER_DIR / f"{name}.csv"
    zip_path = DECODER_DIR / f"{name}.zip"
    save_prediction_rows(pred_rows, csv_path, zip_path)

print("\nCandidate zip files:")
for p in sorted(DECODER_DIR.glob("*.zip")):
    print(" ", p)

In [ ]:
from pathlib import Path
import shutil
import zipfile

src_candidate_dir = Path("/content/prompt_film_mdd/decoder_candidates")
src_work_dir = Path("/content/prompt_film_mdd")

out_dir = Path("/content/mdd_final_artifacts")
out_dir.mkdir(exist_ok=True)

files_to_copy = [
    src_candidate_dir / "ind_strict_conf.zip",
    src_candidate_dir / "ind_dev_tuned_conf.zip",
    src_candidate_dir / "ind_balanced_conf.zip",

    src_work_dir / "best_score.pt",
    src_work_dir / "best_wer.pt",
    src_work_dir / "epoch_020.pt",
    src_work_dir / "epoch_030.pt",
    src_work_dir / "vocab_mdd2025.json",
    src_work_dir / "config.json",
    src_work_dir / "history.json",
    src_work_dir / "dev_predictions_best_wer.csv",
]

for src in files_to_copy:
    if src.exists():
        dst = out_dir / src.name
        shutil.copy2(src, dst)
        print("copied:", dst, "size:", dst.stat().st_size)
    else:
        print("missing:", src)

bundle_zip = Path("/content/mdd_final_artifacts_bundle.zip")
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(out_dir.glob("*")):
        zf.write(file_path, arcname=file_path.name)

print("\nDownload folder:", out_dir)
print("Download bundle:", bundle_zip)
print("Bundle size:", bundle_zip.stat().st_size)

In [ ]:
from pathlib import Path
import shutil
import zipfile

# Mount Drive if needed
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped/failed:", repr(e))

src_candidate_dir = Path("/content/prompt_film_mdd/decoder_candidates")
src_work_dir = Path("/content/prompt_film_mdd")

out_dir = Path("/content/mdd_final_artifacts")
out_dir.mkdir(exist_ok=True)

files_to_copy = [
    src_candidate_dir / "ind_strict_conf.zip",
    src_candidate_dir / "ind_dev_tuned_conf.zip",
    src_candidate_dir / "ind_balanced_conf.zip",

    src_work_dir / "best_score.pt",
    src_work_dir / "best_wer.pt",
    src_work_dir / "epoch_020.pt",
    src_work_dir / "epoch_030.pt",
    src_work_dir / "vocab_mdd2025.json",
    src_work_dir / "config.json",
    src_work_dir / "history.json",
    src_work_dir / "dev_predictions_best_wer.csv",
]

for src in files_to_copy:
    if src.exists():
        dst = out_dir / src.name
        shutil.copy2(src, dst)
        print("copied local:", dst, "size:", dst.stat().st_size)
    else:
        print("missing:", src)

bundle_zip = Path("/content/mdd_final_artifacts_bundle.zip")
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(out_dir.glob("*")):
        zf.write(file_path, arcname=file_path.name)

print("created local bundle:", bundle_zip, "size:", bundle_zip.stat().st_size)

# Copy to Google Drive
drive_dir = Path("/content/drive/MyDrive/mdd_prompt_film_final")
drive_dir.mkdir(parents=True, exist_ok=True)

# Copy bundle
drive_bundle = drive_dir / bundle_zip.name
shutil.copy2(bundle_zip, drive_bundle)
print("copied Drive bundle:", drive_bundle, "size:", drive_bundle.stat().st_size)

# Copy individual files too
for file_path in sorted(out_dir.glob("*")):
    dst = drive_dir / file_path.name
    shutil.copy2(file_path, dst)
    print("copied Drive file:", dst, "size:", dst.stat().st_size)

print("\nGoogle Drive folder:")
print(drive_dir)
print("\nMain bundle:")
print(drive_bundle)